# CASMI 2026 · RL77 Notebook 1 — Two-Stage GLMR (Pre-Retrieval + Spec-Mol Cross-Fusion)

*Assembled from **CASMI 2026 Colab Rubric.pdf** (Gemini conversation, pages 1–21). Python cells are the reviewed `experiments/runs/RL77/nb1/cell_NN.py` files; text cells are taken from the PDF.*

> **User prompt:** Use the @ml-for-chemoinformatics-on-kaggle to create rubric for colab notebooks that will most likely succeed at ranking in the top 10 percentile of the leaderboard for the CASMI 26 challenge on Kaggle

Reaching the top 10 percentile (Bronze medal threshold) in the **Enveda CASMI 2026 - Molecule ID From Mass Spectra** competition requires a public **MRR@25 of at least 0.487 to 0.500+**. Based on the `ml-for-chemoinformatics-on-kaggle` framework, the **GLMR** (Generative Language Model-based Retrieval) architecture, and **Mass2SMILES** spectral-to-structure methodologies, this 100-point rubric evaluates whether a Google Colab training and Kaggle inference workflow is engineered to cross the 0.490+ MRR@25 threshold.

## Summary Scoring Matrix

| Rubric Domain | Max Points | Top 10% Target Benchmark | Primary Failure Mode |
| --- | --- | --- | --- |
| 1. Validation & Metric Alignment | 15 | Local-to-Public gap under 0.03 | Random spectrum split leakage |
| 2. Spectral Curation & Harmonization | 15 | 256-d sinusoidal + neutral losses | Unfiltered noise and adduct drift |
| 3. Formula & Substructure Constraints | 15 | Mass + 60 functional group priors | Unconstrained isomer explosion |
| 4. Multi-Spectrum Hybrid Modeling | 30 | Two-stage Retrieval + Cross-Fusion | Single-spectrum or sparse de novo |
| 5. Post-Processing & Deduplication | 15 | 25 unique InChIKey14 per molecule | Duplicate tautomers or stereoisomers |
| 6. Colab-to-Kaggle Orchestration | 10 | Offline runtime under 9 hours | OOM crash or internet dependency |

The table above weights the six engineering pillars required to bridge the spectral-to-structural modality gap while respecting Kaggle's offline execution limits.

## Readiness Grading Scale

- **90 to 100 Points (Top 10% / Medal Contender):** Fully aligned with 0.490+ MRR@25 public pipelines; combines leakage-free `inchikey14` validation, multi-spectrum aggregation, two-stage GLMR cross-fusion re-ranking, and strict `inchikey14` deduplication.
- **75 to 89 Points (Top 25–30% / Competitive Baseline):** Strong single-stage retrieval or library matching (typical public MRR@25 of 0.450 to 0.480), but missing either context-aware generative re-ranking, multi-spectrum pooling, or tautomer deduplication.
- **Below 75 Points (High Shakeup / Failure Risk):** Vulnerable to severe local-to-public leaderboard divergence (MRR@25 below 0.250) due to random split leakage, unharmonized collision energies, or sparse de novo generation without retrieval priors.

---

> **User prompt:** Generate a complete PyTorch Colab notebook template implementing the two-stage GLMR (Pre-Retrieval + Spec-Mol Cross-Fusion) pipeline and multi-spectrum aggregator for CASMI 2026.

This PyTorch Google Colab notebook template implements the two-stage **GLMR (Generative Language Model-based Retrieval)** architecture paired with a **Multi-Spectrum Attention Aggregator** and **Mass2SMILES** spectral encodings for the **Enveda CASMI 2026** challenge. Each code block corresponds to a sequential Colab cell designed for A100/V100 training and offline Kaggle inference under the 9-hour runtime limit.

## Prerequisites

| Input | Kaggle source | Provides |
| --- | --- | --- |
| Competition data | `enveda-CASMI26-molecule-id-mass-spectra` | `train.parquet`, `test.parquet`, `sample_submission.csv` |
| RL77 prerequisites | `scottweeden/casmi26-rl77-prereqs` | `rl77_env.py`, `rl77_weights.py`, `rl77_weights.json` (Evidently snapshot `rl77-v014`), upstream modules |

- **Order:** This is the first notebook in the chain.
- **Accelerator:** GPU (T4); internet on for the test run so `ensure_packages()` can reach PyPI. Offline runs install from the wheels in notebook 2's output dataset.
- **Packages:** `rdkit==2026.03.3` (the metric's canonicalizer) and `faiss-cpu` are installed if the image lacks them; `torch`, `pyarrow`, `pandas`, `scipy` come from the Kaggle image.
- **Weights:** all blend weights, reward coefficients, and tolerances come from Evidently dataset `casmi-rl77-weights` (version `rl77-v014`, id `01a1ee49-89c9-7ce4-b327-df8b7cf1d48f`) through `W("...")`.
- **Outputs:** everything written to `/kaggle/working` is published as `scottweeden/casmi26-rl77-nb1-glmr-outputs` for the next notebook.

In [ ]:
# Prerequisites: locate the RL77 helper modules, install pinned packages, load the Evidently weights snapshot
import glob, os, sys
from pathlib import Path

_lib_hits = sorted(glob.glob("/kaggle/input/**/rl77_env.py", recursive=True))
if _lib_hits:
    RL77_LIB = Path(_lib_hits[0]).parent
elif os.environ.get("RL77_LIB_DIR"):
    RL77_LIB = Path(os.environ["RL77_LIB_DIR"])  # e.g. experiments/runs/RL77/build/datasets/casmi26-rl77-prereqs
else:
    raise FileNotFoundError("Attach the scottweeden/casmi26-rl77-prereqs dataset or set RL77_LIB_DIR.")
sys.path.insert(0, str(RL77_LIB))

import rl77_env as ENV
ENV.ensure_packages()                      # rdkit==2026.03.3 + faiss-cpu (offline wheels when attached)
from rl77_weights import load_rl77_weights
W = load_rl77_weights()                    # every blend weight / reward coefficient in this notebook reads W("...")

ENV.log(f"Competition data: {ENV.competition_dir()} | outputs: {ENV.work_dir()}")


## Cell 1: Environment Setup, Dependencies & Global Configuration

> **Review fix:** `WEIGHT_decay` was misspelled while Cell 9 read `CFG.WEIGHT_DECAY` (AttributeError). Google Drive paths are replaced by `rl77_env` lookups (competition mount, `/kaggle/working`). Retrieval/beam/temperature values come from the Evidently weights snapshot `W`. The 300/30 epoch counts remain upper bounds; wall-clock budgets (`RL77_STAGE1_MINUTES`, `RL77_STAGE2_MINUTES`) keep the run inside a 12-hour Kaggle session.

In [ ]:
# [Colab Cell 1] Install pinned competition dependencies & configure environment
# Packages are installed by the prerequisites cell (rl77_env.ensure_packages): offline from the
# wheels bundled by notebook 2 when present, otherwise from PyPI with rdkit==2026.03.3 pinned.

import os
import math
import time
import random
from functools import lru_cache
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from rdkit import Chem, RDLogger
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem import inchi

RDLogger.DisableLog("rdApp.*")


class CFG:
    # Paths: competition parquet files and this notebook's output folder (/kaggle/working on Kaggle)
    DATA_DIR = ENV.competition_dir()
    OUTPUT_DIR = ENV.work_dir()
    CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
    SUBMISSION_PATH = OUTPUT_DIR / "submission.csv"
    # Optional pre-trained ChemFormer encoder state_dict (GLMR freezes a pre-trained encoder).
    MOL_ENCODER_CKPT = os.environ.get("RL77_MOL_ENCODER_CKPT", "")

    # Spectral preprocessing constants
    REL_INTENSITY_FLOOR = 0.01       # Drop peaks < 1% of base peak
    PRECURSOR_CLIP_OFFSET = 2.0      # Clip fragments > precursor_mz + 2.0 Da
    C13_MASS_DIFF = 1.003355         # Carbon-13 isotope spacing (Da)
    C13_TOLERANCE_DA = 0.015         # Deisotoping tolerance window
    MAX_PEAKS = 61                   # GLMR spectral encoder sequence length (60 fragments + 1 precursor)
    ELECTRON_MASS_CORR = 0.000548    # Positive-mode electron mass correction for train.parquet

    # GLMR & Multi-Spectrum Architecture Hyperparameters (Appendix B of GLMR)
    D_MODEL = 256
    N_HEADS = 8
    MOL_ENC_LAYERS = 4               # ChemFormer bidirectional encoder layers
    SPEC_ENC_LAYERS = 6              # Spectral Multi-Head Attention layers
    MOL_DEC_LAYERS = 4               # ChemFormer autoregressive decoder layers
    MAX_SMILES_LEN = 256             # Token sequence cap (99.9% of train SMILES are <= 207 chars)
    TOP_K_CANDIDATES = W.int("nb1.retrieval.top_k")          # Pre-retrieval candidate pool size K
    BEAM_SIZE = W.int("nb1.decode.beam_size")                 # Beam search width during generative stage
    NUM_FG_CLASSES = 60              # Mass2SMILES auxiliary functional group head

    # Training Hyperparameters
    SEED = 42
    BATCH_SIZE = 64
    LR = 1e-4
    WEIGHT_DECAY = 0.1
    TEMPERATURE = W("nb1.loss.infonce_temperature")          # Dual-path InfoNCE temperature tau
    EPOCHS_STAGE1 = 300              # Upper bound; the wall-clock budget below usually ends Stage 1 first
    EPOCHS_STAGE2 = 30               # Upper bound for Stage 2
    STAGE1_MINUTES = float(os.environ.get("RL77_STAGE1_MINUTES", 55))
    STAGE2_MINUTES = float(os.environ.get("RL77_STAGE2_MINUTES", 35))
    MASS_TOLERANCE_PPM = W("nb1.retrieval.mass_tolerance_ppm")  # Candidate library mass filter window
    MIN_TOLERANCE_DA = W("nb1.retrieval.min_tolerance_da")
    TOP_N_SUBMISSION = 25            # Required ranked candidates per molecule_id

    # Real-data run sizes (a Kaggle GPU session is capped at 12 hours)
    N_TRAIN_MOLECULES = int(os.environ.get("RL77_N_TRAIN_MOLECULES", 40000))
    N_VAL_MOLECULES = int(os.environ.get("RL77_N_VAL_MOLECULES", 1000))
    N_VAL_EVAL = int(os.environ.get("RL77_N_VAL_EVAL", 250))  # val molecules decoded with beam search
    MAX_SPECTRA_LOADED = 32          # spectra kept per inchikey14 when streaming train.parquet
    MAX_SPECTRA_TRAIN = 8            # spectra sampled per molecule per training step
    MAX_SPECTRA_INFER = 16           # test.parquet has at most 9 spectra per molecule_id
    NUM_WORKERS = min(4, os.cpu_count() or 1)
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_everything(seed: int = CFG.SEED) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything()
CFG.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

## Cell 2: RDKit 2026.03.3 Tautomer Canonicalization, `inchikey14` & SMILES Tokenizer

> **Review fix:** The competition files spell formate `[M+CH2O2-H]-` and acetate `[M+C2H4O2-H]-` (the PDF used `[M+FA-H]-` / `[M+Hac-H]-`), and train has ~340k `[2M+X]` multimer spectra. Unknown adducts silently fell back to the `[M+H]+` shift, mis-estimating neutral mass by ~46 Da (formate) or 2× (multimers). `neutral_mass_from_precursor()` now handles n, z, and delta. The tokenizer vocabulary listed `C` twice and lacked `H` and `%`, so `[nH]`, `[C@H]`, and ring closures ≥10 (in >10% of train structures) decoded as empty strings; the vocabulary is now the measured train character set.

In [ ]:
# [Colab Cell 2] Chemical standardization, evaluation metric (MRR@25), and SMILES vocabulary
tautomer_enumerator = rdMolStandardize.TautomerEnumerator()

PROTON = 1.007276467
ELECTRON = 0.000548580

# Adduct name -> (molecules per ion n, charge z, mass added to n*M to form the ion).
# Names follow train.parquet / test.parquet spelling (formate is "[M+CH2O2-H]-", acetate "[M+C2H4O2-H]-").
ADDUCT_SPECS: Dict[str, Tuple[int, int, float]] = {
    "[M+H]+": (1, 1, PROTON),
    "[M+Na]+": (1, 1, 22.989218),
    "[M+NH4]+": (1, 1, 18.033823),
    "[M+K]+": (1, 1, 38.963158),
    "[M-H2O+H]+": (1, 1, PROTON - 18.010565),
    "[M-2H2O+H]+": (1, 1, PROTON - 2 * 18.010565),
    "[M]+": (1, 1, -ELECTRON),
    "[M-H]-": (1, -1, -PROTON),
    "[M+CH2O2-H]-": (1, -1, 46.005479 - PROTON),
    "[M+C2H4O2-H]-": (1, -1, 60.021129 - PROTON),
    "[M+Cl]-": (1, -1, 34.969402),
    "[2M+H]+": (2, 1, PROTON),
    "[2M+Na]+": (2, 1, 22.989218),
    "[2M-H]-": (2, -1, -PROTON),
    "[M+2H]2+": (1, 2, 2 * PROTON),
    "[M-2H]2-": (1, -2, -2 * PROTON),
}
ADDUCT_VOCAB = {name: idx for idx, name in enumerate(ADDUCT_SPECS)}
ADDUCT_VOCAB["UNKNOWN"] = len(ADDUCT_VOCAB)

# Shifts for singly charged monomer adducts, kept for callers that use neutral = precursor_mz + shift.
ADDUCT_MASS_SHIFT = {name: -delta for name, (n, z, delta) in ADDUCT_SPECS.items() if n == 1 and abs(z) == 1}


def neutral_mass_from_precursor(precursor_mz: float, adduct: str, ionization_mode: str = "positive") -> float:
    """Neutral monoisotopic mass M from precursor m/z: M = (|z| * m/z - delta) / n."""
    spec = ADDUCT_SPECS.get(str(adduct))
    if spec is None:
        spec = ADDUCT_SPECS["[M-H]-"] if str(ionization_mode).lower().startswith("neg") else ADDUCT_SPECS["[M+H]+"]
    n, z, delta = spec
    return (abs(z) * float(precursor_mz) - delta) / n


@lru_cache(maxsize=2_000_000)
def canonicalize_and_inchikey14(smiles: str) -> Tuple[Optional[str], Optional[str]]:
    """
    Passes SMILES through RDKit tautomer canonicalization and extracts
    the 14-character planar connectivity block (InChIKey14). Cached: the same
    candidate SMILES is scored many times across molecules and ranking passes.
    """
    if not smiles or not isinstance(smiles, str):
        return None, None
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, None
    try:
        canon_mol = tautomer_enumerator.Canonicalize(mol)
        canon_smiles = Chem.MolToSmiles(canon_mol, isomericSmiles=False, canonical=True)
        ik = inchi.MolToInchiKey(canon_mol)
        if not ik or len(ik) < 14:
            return None, None
        return canon_smiles, ik[:14]
    except Exception:
        return None, None


def compute_casmi_mrr25(
    predictions: Dict[str, List[str]],
    ground_truth_ik14: Dict[str, str]
) -> float:
    """Computes official first-match MRR@25 across molecules using InChIKey14."""
    reciprocal_ranks = []
    for mol_id, true_ik14 in ground_truth_ik14.items():
        preds = predictions.get(mol_id, [])[:CFG.TOP_N_SUBMISSION]
        rr = 0.0
        for rank_idx, smi in enumerate(preds, start=1):
            _, pred_ik14 = canonicalize_and_inchikey14(smi)
            if pred_ik14 is not None and pred_ik14 == true_ik14:
                rr = 1.0 / rank_idx
                break
        reciprocal_ranks.append(rr)
    return float(np.mean(reciprocal_ranks)) if reciprocal_ranks else 0.0


class SMILESTokenizer:
    """Character tokenizer with [PAD], [CLS], [BOS], [EOS], and [UNK] tokens.

    CHARS is the character set of train.parquet normalized_smiles (no stereo), plus the
    stereo/bond characters so isomeric inputs do not collapse to [UNK]. Two-letter
    elements (Cl, Br, Si, As, Se, Na) are spelled as two character tokens.
    """
    SPECIAL_TOKENS = ["[PAD]", "[CLS]", "[BOS]", "[EOS]", "[UNK]"]
    CHARS = list("#%()+-0123456789=ABCFHIKNOPS[]aceilnoprsg@/\\.")

    def __init__(self, max_len: int = CFG.MAX_SMILES_LEN):
        assert len(set(self.CHARS)) == len(self.CHARS), "duplicate characters in SMILES vocabulary"
        self.max_len = max_len
        self.vocab = self.SPECIAL_TOKENS + self.CHARS
        self.stoi = {ch: idx for idx, ch in enumerate(self.vocab)}
        self.itos = {idx: ch for idx, ch in enumerate(self.vocab)}
        self.pad_id = self.stoi["[PAD]"]
        self.cls_id = self.stoi["[CLS]"]
        self.bos_id = self.stoi["[BOS]"]
        self.eos_id = self.stoi["[EOS]"]
        self.unk_id = self.stoi["[UNK]"]

    def __len__(self) -> int:
        return len(self.vocab)

    def encode(self, smiles: str, add_cls: bool = True) -> torch.Tensor:
        tokens = [self.cls_id] if add_cls else [self.bos_id]
        for ch in smiles[: self.max_len - 2]:
            tokens.append(self.stoi.get(ch, self.unk_id))
        tokens.append(self.eos_id)
        pad_len = self.max_len - len(tokens)
        if pad_len > 0:
            tokens.extend([self.pad_id] * pad_len)
        return torch.tensor(tokens[: self.max_len], dtype=torch.long)

    def decode(self, token_ids: List[int]) -> str:
        chars = []
        for idx in token_ids:
            idx = int(idx)
            if idx == self.eos_id:
                break
            if idx in (self.pad_id, self.cls_id, self.bos_id, self.unk_id):
                continue
            chars.append(self.itos.get(idx, ""))
        return "".join(chars)

## Cell 3: Spectral Curation, Electron-Mass Alignment & Neutral Loss Encoding

In [ ]:
# [Colab Cell 3] Spectral preprocessing: electron-mass correction, deisotoping, clipping, & neutral losses
def preprocess_spectrum(
    mzs: np.ndarray,
    intensities: np.ndarray,
    precursor_mz: float,
    ionization_mode: str = "positive",
    apply_electron_mass_correction: bool = False,
    max_peaks: int = CFG.MAX_PEAKS,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Cleans raw MS/MS peak lists and extracts (mz, sqrt_intensity, neutral_loss) tuples.
    Following Mass2SMILES & GLMR:
      - Precursor peak is placed at index 0 with relative intensity = 2.0.
      - Up to (max_peaks - 1) fragment ions follow in descending intensity order.
    """
    mzs = np.asarray(mzs, dtype=np.float64).copy()
    intensities = np.asarray(intensities, dtype=np.float64).copy()

    # 1. Harmonize positive-mode electron mass on training sets (e.g., enveda-180) with hidden test set
    if apply_electron_mass_correction and str(ionization_mode).lower() == "positive":
        mzs -= CFG.ELECTRON_MASS_CORR
        precursor_mz -= CFG.ELECTRON_MASS_CORR

    if len(mzs) == 0:
        out_mzs = np.zeros(max_peaks, dtype=np.float32)
        out_ints = np.zeros(max_peaks, dtype=np.float32)
        out_nls = np.zeros(max_peaks, dtype=np.float32)
        out_mzs[0] = precursor_mz
        out_ints[0] = 2.0
        return out_mzs, out_ints, out_nls

    # 2. Normalize to base peak = 1.0 & apply relative-intensity floor (1%)
    max_int = np.max(intensities)
    if max_int > 0:
        intensities = intensities / max_int
    valid_mask = (intensities >= CFG.REL_INTENSITY_FLOOR) & (
        mzs <= (precursor_mz + CFG.PRECURSOR_CLIP_OFFSET)
    )
    mzs, intensities = mzs[valid_mask], intensities[valid_mask]

    # 3. Carbon-13 deisotoping (~1.003355 Da above more intense parent fragment)
    if len(mzs) > 1:
        order = np.argsort(mzs)
        mzs, intensities = mzs[order], intensities[order]
        keep = np.ones(len(mzs), dtype=bool)
        for i in range(len(mzs)):
            if not keep[i]:
                continue
            diffs = mzs[i + 1 :] - mzs[i]
            iso_hits = np.where(
                (np.abs(diffs - CFG.C13_MASS_DIFF) <= CFG.C13_TOLERANCE_DA)
                & (intensities[i + 1 :] < intensities[i] * 0.8)
            )[0]
            keep[i + 1 + iso_hits] = False
        mzs, intensities = mzs[keep], intensities[keep]

    # 4. Top-(max_peaks - 1) selection + square-root scaling
    n_frag = max_peaks - 1
    if len(mzs) > n_frag:
        top_idx = np.argsort(intensities)[::-1][:n_frag]
        top_idx = top_idx[np.argsort(mzs[top_idx])]
        mzs, intensities = mzs[top_idx], intensities[top_idx]

    sqrt_ints = np.sqrt(np.clip(intensities, 0.0, 1.0))
    neutral_losses = np.clip(precursor_mz - mzs, 0.0, None)

    # 5. Pad to fixed length with precursor ion anchored at index 0 (intensity = 2.0)
    out_mzs = np.zeros(max_peaks, dtype=np.float32)
    out_ints = np.zeros(max_peaks, dtype=np.float32)
    out_nls = np.zeros(max_peaks, dtype=np.float32)

    out_mzs[0] = float(precursor_mz)
    out_ints[0] = 2.0
    out_nls[0] = 0.0

    k = min(len(mzs), n_frag)
    if k > 0:
        out_mzs[1 : 1 + k] = mzs[:k]
        out_ints[1 : 1 + k] = sqrt_ints[:k]
        out_nls[1 : 1 + k] = neutral_losses[:k]

    return out_mzs, out_ints, out_nls


def harmonize_collision_energy(ce_entry) -> float:
    """Converts single or multi-energy list (e.g., [20, 40, 60, 80]) into mean eV."""
    if ce_entry is None:
        return 35.0
    if isinstance(ce_entry, (list, tuple, np.ndarray)):
        vals = [float(x) for x in ce_entry if x is not None and not np.isnan(float(x))]
        return float(np.mean(vals)) if vals else 35.0
    try:
        val = float(ce_entry)
        return 35.0 if np.isnan(val) else val
    except Exception:
        return 35.0

## Cell 4: Leakage-Free `inchikey14` Grouped Dataset & Multi-Spectrum Collation

> **Review fix:** 46% of `train.parquet` `base_peak_intensity` values are null and `max(nan, 1.0)` returns `nan`, which propagated NaN into every encoder pass; non-finite values now default to 1e4. The primary precursor is taken from a singly charged monomer adduct. `load_competition_frames()` applies the same leakage rule as `create_leakage_free_splits()` while streaming the 3 GB parquet through pyarrow filters instead of pandas.

In [ ]:
# [Colab Cell 4] Multi-Spectrum Molecule Dataset & Leakage-Free InChIKey14 Split
def create_leakage_free_splits(
    df: pd.DataFrame, val_fraction: float = 0.10, seed: int = CFG.SEED
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Splits training data so zero inchikey14 connectivity blocks overlap between train and val,
    while prioritizing timsTOF spectra (157-1159 Da) for validation alignment with test.parquet.
    """
    rng = np.random.default_rng(seed)
    timstof_mask = (
        (df["instrument_type"] == "timsTOF")
        & (df["precursor_mz"] >= 157.0)
        & (df["precursor_mz"] <= 1159.0)
    )
    eligible_ik14 = df.loc[timstof_mask, "inchikey14"].dropna().unique()
    if len(eligible_ik14) == 0:
        eligible_ik14 = df["inchikey14"].dropna().unique()

    n_val = max(1, int(len(eligible_ik14) * val_fraction))
    val_ik14_set = set(rng.choice(eligible_ik14, size=n_val, replace=False))

    val_df = df[df["inchikey14"].isin(val_ik14_set) & timstof_mask].reset_index(drop=True)
    train_df = df[~df["inchikey14"].isin(val_ik14_set)].reset_index(drop=True)
    return train_df, val_df


TRAIN_COLUMNS = [
    "inchikey14", "normalized_smiles", "molecular_formula", "ingest_lib", "instrument_type",
    "ionization_mode", "adduct", "precursor_mz", "ms2_mzs", "ms2_normalized_intensities",
    "base_peak_intensity", "collision_energy_ev",
]


def load_competition_frames(
    data_dir: Path,
    n_train_mols: int,
    n_val_mols: int,
    seed: int = CFG.SEED,
    max_spectra_per_mol: int = CFG.MAX_SPECTRA_LOADED,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Streams train.parquet (2.5M spectra, 3 GB) with the same leakage rule as
    create_leakage_free_splits, without materializing the whole file in pandas:
      - validation inchikey14 blocks are sampled from timsTOF spectra at 157-1159 Da,
      - those blocks are removed from training entirely,
      - training blocks are sampled from the rest.
    Returns (train_df, val_df, library_df) where library_df holds one SMILES per inchikey14
    over the whole file (the retrieval library used for validation and test).
    """
    import pyarrow.compute as pc

    path = Path(data_dir) / "train.parquet"
    keys = pq.read_table(path, columns=["inchikey14", "normalized_smiles", "instrument_type", "precursor_mz"])
    timstof = pc.and_(
        pc.equal(keys["instrument_type"], "timsTOF"),
        pc.and_(pc.greater_equal(keys["precursor_mz"], 157.0), pc.less_equal(keys["precursor_mz"], 1159.0)),
    )
    eligible = np.array(sorted(pc.unique(pc.filter(keys["inchikey14"], timstof)).to_pylist()))
    all_ik14 = np.array(sorted(pc.unique(keys["inchikey14"]).to_pylist()))

    rng = np.random.default_rng(seed)
    val_ik14 = set(rng.choice(eligible, size=min(n_val_mols, len(eligible)), replace=False).tolist())
    remaining = np.array([k for k in all_ik14 if k not in val_ik14])
    train_ik14 = set(rng.choice(remaining, size=min(n_train_mols, len(remaining)), replace=False).tolist())

    lib = (
        keys.select(["inchikey14", "normalized_smiles"]).to_pandas()
        .rename(columns={"normalized_smiles": "smiles"})
        .dropna().drop_duplicates("inchikey14").reset_index(drop=True)
    )
    del keys

    selected = sorted(val_ik14 | train_ik14)
    table = pq.read_table(path, columns=TRAIN_COLUMNS, filters=[("inchikey14", "in", selected)])
    df = table.to_pandas()
    del table
    df = df.groupby("inchikey14", sort=False).head(max_spectra_per_mol).reset_index(drop=True)

    in_window = (
        (df["instrument_type"] == "timsTOF") & (df["precursor_mz"] >= 157.0) & (df["precursor_mz"] <= 1159.0)
    )
    val_df = df[df["inchikey14"].isin(val_ik14) & in_window].reset_index(drop=True)
    train_df = df[df["inchikey14"].isin(train_ik14)].reset_index(drop=True)
    assert not set(train_df["inchikey14"]) & set(val_df["inchikey14"]), "inchikey14 leakage between train and val"
    ENV.log(
        f"Loaded {train_df['inchikey14'].nunique():,} train molecules ({len(train_df):,} spectra), "
        f"{val_df['inchikey14'].nunique():,} val molecules ({len(val_df):,} spectra), library {len(lib):,} structures"
    )
    return train_df, val_df, lib


def load_spectra_for_inchikeys(
    data_dir: Path,
    inchikeys: List[str],
    timstof_window_only: bool = False,
    max_spectra_per_mol: int = CFG.MAX_SPECTRA_LOADED,
) -> pd.DataFrame:
    """Reloads the spectra of a saved split (notebook 1 `split/*.txt`) so notebooks 2-5 use the same molecules."""
    table = pq.read_table(Path(data_dir) / "train.parquet", columns=TRAIN_COLUMNS,
                          filters=[("inchikey14", "in", sorted(set(inchikeys)))])
    df = table.to_pandas()
    if timstof_window_only:
        df = df[(df["instrument_type"] == "timsTOF") & (df["precursor_mz"] >= 157.0) & (df["precursor_mz"] <= 1159.0)]
    df = df.dropna(subset=["normalized_smiles"])
    return df.groupby("inchikey14", sort=False).head(max_spectra_per_mol).reset_index(drop=True)


def read_split(split_dir: Path, name: str) -> List[str]:
    return [line.strip() for line in (Path(split_dir) / f"{name}_inchikey14.txt").read_text().splitlines() if line.strip()]


def ground_truth_ik14(df: pd.DataFrame) -> Dict[str, str]:
    """Validation answer key: molecule_id -> InChIKey14 of the tautomer-canonical true structure.

    train.parquet has no molecule_id, so validation groups (and records) are keyed by its stored inchikey14. The scorer
    compares tautomer-canonical InChIKey14s, and 0.3% of the held-out molecules (1.5% of train) store the key of a
    non-canonical tautomer, which `{mid: mid}` made unmatchable even for a correct answer."""
    first = df.drop_duplicates("inchikey14")
    return {str(mid): (canonicalize_and_inchikey14(smi)[1] or str(mid))
            for mid, smi in zip(first["inchikey14"], first["normalized_smiles"])}


def _finite_or(value, default: float) -> float:
    try:
        value = float(value)
    except (TypeError, ValueError):
        return default
    return value if math.isfinite(value) else default


def _primary_record_index(records: List[dict]) -> int:
    """First spectrum whose adduct is a known singly charged monomer, so precursor_mz gives M directly."""
    for i, row in enumerate(records):
        spec = ADDUCT_SPECS.get(str(row.get("adduct", "")))
        if spec is not None and spec[0] == 1 and abs(spec[1]) == 1:
            return i
    return 0


class MultiSpectrumMoleculeDataset(Dataset):
    """
    Groups all MS/MS spectra belonging to the same molecule_id (1 to 16 spectra per molecule)
    for joint multi-spectrum encoding and contrastive/generative training.
    """

    def __init__(
        self,
        df: pd.DataFrame,
        tokenizer: SMILESTokenizer,
        max_spectra_per_mol: int = 8,
        is_train: bool = True,
    ):
        self.tokenizer = tokenizer
        self.max_spectra_per_mol = max_spectra_per_mol
        self.is_train = is_train

        group_col = "molecule_id" if "molecule_id" in df.columns else "inchikey14"
        self.groups = []
        for mol_id, grp in df.groupby(group_col, sort=False):
            self.groups.append((str(mol_id), grp.to_dict("records")))

    def __len__(self) -> int:
        return len(self.groups)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        mol_id, records = self.groups[idx]
        if self.is_train and len(records) > self.max_spectra_per_mol:
            records = random.sample(records, self.max_spectra_per_mol)
        else:
            records = records[: self.max_spectra_per_mol]

        mzs_list, ints_list, nls_list, ce_list, adduct_list, bpi_list = [], [], [], [], [], []
        for row in records:
            cached = row.get("_preprocessed")
            if cached is None:
                is_enveda = str(row.get("ingest_lib", "")) == "enveda-180"
                cached = preprocess_spectrum(
                    row["ms2_mzs"],
                    row["ms2_normalized_intensities"],
                    float(row["precursor_mz"]),
                    ionization_mode=str(row.get("ionization_mode", "positive")),
                    apply_electron_mass_correction=(self.is_train and is_enveda),
                )
                row["_preprocessed"] = cached  # per-worker cache; DataLoader workers are persistent
            mzs, ints, nls = cached
            mzs_list.append(mzs)
            ints_list.append(ints)
            nls_list.append(nls)
            ce_list.append(harmonize_collision_energy(row.get("collision_energy_ev", 35.0)))
            adduct_list.append(ADDUCT_VOCAB.get(str(row.get("adduct", "UNKNOWN")), ADDUCT_VOCAB["UNKNOWN"]))
            # 46% of train.parquet base_peak_intensity values are null; max(nan, 1.0) is nan, so guard first.
            bpi = _finite_or(row.get("base_peak_intensity"), 1e4)
            bpi_list.append(math.log10(max(bpi, 1.0)))

        # Pad spectrum count up to max_spectra_per_mol
        n_spec = len(mzs_list)
        spec_mask = np.zeros(self.max_spectra_per_mol, dtype=bool)
        spec_mask[:n_spec] = True

        while len(mzs_list) < self.max_spectra_per_mol:
            mzs_list.append(np.zeros(CFG.MAX_PEAKS, dtype=np.float32))
            ints_list.append(np.zeros(CFG.MAX_PEAKS, dtype=np.float32))
            nls_list.append(np.zeros(CFG.MAX_PEAKS, dtype=np.float32))
            ce_list.append(0.0)
            adduct_list.append(ADDUCT_VOCAB["UNKNOWN"])
            bpi_list.append(0.0)

        item = {
            "molecule_id": mol_id,
            "mzs": torch.tensor(np.stack(mzs_list), dtype=torch.float32),          # (S, P)
            "ints": torch.tensor(np.stack(ints_list), dtype=torch.float32),        # (S, P)
            "nls": torch.tensor(np.stack(nls_list), dtype=torch.float32),          # (S, P)
            "ce": torch.tensor(ce_list, dtype=torch.float32),                      # (S,)
            "adduct": torch.tensor(adduct_list, dtype=torch.long),                 # (S,)
            "bpi": torch.tensor(bpi_list, dtype=torch.float32),                    # (S,)
            "spec_mask": torch.tensor(spec_mask, dtype=torch.bool),                # (S,)
        }
        primary = records[_primary_record_index(records)]
        precursor_mz = float(primary["precursor_mz"])
        if self.is_train and str(primary.get("ingest_lib", "")) == "enveda-180" \
                and str(primary.get("ionization_mode", "positive")).lower() == "positive":
            precursor_mz -= CFG.ELECTRON_MASS_CORR  # same correction preprocess_spectrum applied
        item["precursor_mz"] = torch.tensor(precursor_mz, dtype=torch.float64)
        item["primary_adduct"] = str(primary.get("adduct", "[M+H]+"))
        item["ionization_mode"] = str(primary.get("ionization_mode", "positive"))

        smiles = records[0].get("normalized_smiles")
        if isinstance(smiles, str) and smiles:
            item["mol_tokens_cls"] = self.tokenizer.encode(smiles, add_cls=True)
            item["mol_tokens_dec"] = self.tokenizer.encode(smiles, add_cls=False)
            item["smiles"] = smiles
            item["inchikey14"] = str(records[0].get("inchikey14", ""))
            item["molecular_formula"] = str(records[0].get("molecular_formula", "") or "")

        return item

## Cell 5: GLMR Architecture & Multi-Spectrum Attention Aggregator

> **Review fix:** The decoder mixed a float causal mask with a boolean padding mask (deprecated in PyTorch). Beam search ran one forward pass per beam; live beams now decode in one batched pass per step (same search, ~beam-size× fewer passes).

In [ ]:
# [Colab Cell 5] Complete PyTorch GLMR Architecture + Multi-Spectrum Aggregator
class MultiScaleSinusoidalMzEmbedding(nn.Module):
    """
    256-dimensional multi-scale sinusoidal encoding for high-resolution m/z & neutral losses
    (2-decimal precision, Voronov et al. / Mass2SMILES).
    """

    def __init__(self, d_model: int = CFG.D_MODEL, min_freq: float = 1e-2, max_freq: float = 1e3):
        super().__init__()
        assert d_model % 2 == 0
        freqs = torch.logspace(
            math.log10(min_freq), math.log10(max_freq), d_model // 2
        )
        self.register_buffer("freqs", freqs * 2.0 * math.pi)

    def forward(self, mz: torch.Tensor) -> torch.Tensor:
        # mz: (..., P) -> returns (..., P, d_model)
        angles = mz.unsqueeze(-1) * self.freqs
        return torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1)


class SinusoidalSpectralEncoder(nn.Module):
    """
    GLMR Spectral Encoder f_s(Spec; eta):
      - Projects 256-d sinusoidal m/z + 256-d sinusoidal neutral loss + intensity + metadata
      - Passes through 6 Multi-Head Attention encoder layers (d_model = 256)
      - Outputs sequence representations H^s and mean-pooled spectrum embedding E^s
    """

    def __init__(
        self,
        d_model: int = CFG.D_MODEL,
        n_heads: int = CFG.N_HEADS,
        num_layers: int = CFG.SPEC_ENC_LAYERS,
    ):
        super().__init__()
        self.sin_emb = MultiScaleSinusoidalMzEmbedding(d_model)
        self.adduct_emb = nn.Embedding(len(ADDUCT_VOCAB), 32)
        self.ce_proj = nn.Linear(2, 32)  # [collision_energy_ev / 100.0, log10(BPI) / 10.0]

        # Input per peak: 256 (m/z) + 256 (neutral loss) + 1 (intensity) + 64 (metadata) = 577
        self.input_proj = nn.Sequential(
            nn.Linear(d_model * 2 + 1 + 64, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=0.1,
            activation="gelu",
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(
        self,
        mzs: torch.Tensor,
        ints: torch.Tensor,
        nls: torch.Tensor,
        ce: torch.Tensor,
        adduct: torch.Tensor,
        bpi: torch.Tensor,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # Inputs: (B_spec, P)
        mz_feat = self.sin_emb(mzs)
        nl_feat = self.sin_emb(nls)
        meta = torch.cat(
            [
                self.adduct_emb(adduct),
                self.ce_proj(torch.stack([ce / 100.0, bpi / 10.0], dim=-1)),
            ],
            dim=-1,
        )
        meta_expanded = meta.unsqueeze(1).expand(-1, mzs.size(1), -1)

        x = torch.cat([mz_feat, nl_feat, ints.unsqueeze(-1), meta_expanded], dim=-1)
        h_s = self.input_proj(x)

        peak_padding_mask = mzs <= 0.0
        peak_padding_mask[:, 0] = False  # Precursor token at index 0 is always valid
        h_s = self.transformer(h_s, src_key_padding_mask=peak_padding_mask)

        valid_counts = (~peak_padding_mask).float().sum(dim=1, keepdim=True).clamp(min=1.0)
        masked_h_s = h_s.masked_fill(peak_padding_mask.unsqueeze(-1), 0.0)
        e_s = masked_h_s.sum(dim=1) / valid_counts
        return h_s, F.normalize(e_s, p=2, dim=-1)


class MultiSpectrumMoleculeAggregator(nn.Module):
    """
    Aggregates variable numbers of spectra (1..S) per molecule_id across collision energies
    and adducts using learned attention weights + signal-quality gating.
    """

    def __init__(self, d_model: int = CFG.D_MODEL, n_heads: int = CFG.N_HEADS):
        super().__init__()
        self.spec_attn = nn.MultiheadAttention(d_model, n_heads, dropout=0.1, batch_first=True)
        self.score_gate = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, 1),
        )
        self.norm = nn.LayerNorm(d_model)
        self.fg_head = nn.Linear(d_model, CFG.NUM_FG_CLASSES)  # Mass2SMILES auxiliary head

    def forward(
        self,
        h_s_all: torch.Tensor,   # (B, S, P, D)
        e_s_all: torch.Tensor,   # (B, S, D)
        spec_mask: torch.Tensor, # (B, S) True where spectrum is valid
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        B, S, P, D = h_s_all.shape
        key_padding_mask = ~spec_mask

        # Cross-spectrum self-attention over spectrum-level embeddings
        attn_out, _ = self.spec_attn(
            e_s_all, e_s_all, e_s_all, key_padding_mask=key_padding_mask
        )
        e_s_refined = self.norm(e_s_all + attn_out)

        # Compute attention weights alpha_s over the S spectra of each molecule
        logits = self.score_gate(e_s_refined).squeeze(-1)  # (B, S)
        logits = logits.masked_fill(key_padding_mask, -1e9)
        weights = F.softmax(logits, dim=-1)                # (B, S)

        # Unified molecule-level spectral embedding E^s_mol: (B, D)
        e_mol = torch.sum(e_s_refined * weights.unsqueeze(-1), dim=1)
        e_mol = F.normalize(e_mol, p=2, dim=-1)

        # Weighted pooling of peak token states H^s_mol: (B, P, D) for Stage 2 Cross-Fusion
        h_mol = torch.sum(h_s_all * weights.view(B, S, 1, 1), dim=1)
        fg_pred = self.fg_head(e_mol)
        return h_mol, e_mol, fg_pred


class ChemFormerMolecularEncoder(nn.Module):
    """
    GLMR Molecular Encoder f_m(Mol; gamma):
      - 4 bidirectional encoder layers (d_model = 256)
      - [CLS] token at index 0 produces global molecular embedding E^m = H^m_[CLS]
    """

    def __init__(
        self,
        vocab_size: int,
        pad_id: int,
        d_model: int = CFG.D_MODEL,
        n_heads: int = CFG.N_HEADS,
        num_layers: int = CFG.MOL_ENC_LAYERS,
        max_len: int = CFG.MAX_SMILES_LEN,
    ):
        super().__init__()
        self.pad_id = pad_id
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_emb = nn.Embedding(max_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=0.1,
            activation="gelu",
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, input_ids: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        B, L = input_ids.shape
        positions = torch.arange(L, device=input_ids.device).unsqueeze(0).expand(B, L)
        x = self.token_emb(input_ids) + self.pos_emb(positions)
        pad_mask = input_ids == self.pad_id
        h_m = self.encoder(x, src_key_padding_mask=pad_mask)
        e_m = F.normalize(h_m[:, 0, :], p=2, dim=-1)  # [CLS] embedding
        return h_m, e_m


class SpecMolCrossFusion(nn.Module):
    """
    GLMR Equation (7):
    H^ca = Attn(Query(H^s), Key(H^m_K)) * Value(H^m_K)
    Integrates the multi-spectrum representation H^s (Query) with the top-K
    pre-retrieved molecular candidate embeddings H^m_K (Key, Value).
    """

    def __init__(self, d_model: int = CFG.D_MODEL, n_heads: int = CFG.N_HEADS):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(d_model, n_heads, dropout=0.1, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(d_model * 4, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, h_s: torch.Tensor, h_m_k: torch.Tensor) -> torch.Tensor:
        # h_s: (B, P, D), h_m_k: (B, K, D)
        attn_out, _ = self.cross_attn(query=h_s, key=h_m_k, value=h_m_k)
        x = self.norm1(h_s + attn_out)
        h_ca = self.norm2(x + self.ffn(x))
        return h_ca


class ChemFormerMolecularDecoder(nn.Module):
    """
    GLMR Molecular Decoder f_g(H^ca; phi):
      - 4 autoregressive Transformer decoder layers conditioned on H^ca
      - Supports teacher-forced training and Beam Search (beam_size=5) inference
    """

    def __init__(
        self,
        vocab_size: int,
        pad_id: int,
        bos_id: int,
        eos_id: int,
        d_model: int = CFG.D_MODEL,
        n_heads: int = CFG.N_HEADS,
        num_layers: int = CFG.MOL_DEC_LAYERS,
        max_len: int = CFG.MAX_SMILES_LEN,
    ):
        super().__init__()
        self.pad_id = pad_id
        self.bos_id = bos_id
        self.eos_id = eos_id
        self.max_len = max_len

        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_emb = nn.Embedding(max_len, d_model)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=0.1,
            activation="gelu",
            batch_first=True,
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.out_proj = nn.Linear(d_model, vocab_size)

    def forward(self, tgt_ids: torch.Tensor, memory: torch.Tensor) -> torch.Tensor:
        B, Q = tgt_ids.shape
        positions = torch.arange(Q, device=tgt_ids.device).unsqueeze(0).expand(B, Q)
        tgt = self.token_emb(tgt_ids) + self.pos_emb(positions)
        # Boolean causal mask: matches the boolean key-padding mask (mixing float and bool masks is deprecated).
        causal_mask = torch.triu(torch.ones(Q, Q, dtype=torch.bool, device=tgt_ids.device), diagonal=1)
        tgt_pad_mask = tgt_ids == self.pad_id
        dec_out = self.decoder(
            tgt=tgt,
            memory=memory,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=tgt_pad_mask,
        )
        return self.out_proj(dec_out)

    @torch.no_grad()
    def beam_search_decode(
        self, memory: torch.Tensor, beam_size: int = CFG.BEAM_SIZE, max_len: int = 128
    ) -> List[List[int]]:
        """Generates refined SMILES sequences via beam search for each item in batch.

        All live beams of a molecule share one length, so they are decoded in a single
        batched forward pass per step instead of one pass per beam.
        """
        B = memory.size(0)
        max_len = min(max_len, self.max_len)
        best_sequences = []
        for b in range(B):
            mem_b = memory[b : b + 1]  # (1, P, D)
            beams = [([self.bos_id], 0.0)]
            completed = []

            for _ in range(max_len - 1):
                live = []
                for seq, score in beams:
                    (completed if seq[-1] == self.eos_id else live).append((seq, score))
                beams = live
                if not beams or len(completed) >= beam_size:
                    break
                inp = torch.tensor([seq for seq, _ in beams], dtype=torch.long, device=memory.device)
                logits = self.forward(inp, mem_b.expand(len(beams), -1, -1))[:, -1, :]
                log_probs = F.log_softmax(logits, dim=-1)
                topk_lp, topk_idx = torch.topk(log_probs, beam_size, dim=-1)
                candidates = []
                for i, (seq, score) in enumerate(beams):
                    for lp, tok_id in zip(topk_lp[i].tolist(), topk_idx[i].tolist()):
                        candidates.append((seq + [tok_id], score + lp))
                candidates.sort(key=lambda x: x[1] / len(x[0]), reverse=True)
                beams = candidates[:beam_size]

            all_finished = completed + beams
            all_finished.sort(key=lambda x: x[1] / len(x[0]), reverse=True)
            best_sequences.append(all_finished[0][0])
        return best_sequences


class GLMRPipeline(nn.Module):
    """Unified wrapper holding Stage 1 and Stage 2 modules."""

    def __init__(self, tokenizer: SMILESTokenizer):
        super().__init__()
        self.mol_encoder = ChemFormerMolecularEncoder(
            vocab_size=len(tokenizer), pad_id=tokenizer.pad_id
        )
        self.spec_encoder = SinusoidalSpectralEncoder()
        self.aggregator = MultiSpectrumMoleculeAggregator()
        self.cross_fusion = SpecMolCrossFusion()
        self.mol_decoder = ChemFormerMolecularDecoder(
            vocab_size=len(tokenizer),
            pad_id=tokenizer.pad_id,
            bos_id=tokenizer.bos_id,
            eos_id=tokenizer.eos_id,
        )

    def encode_molecule_spectra(self, batch: Dict[str, torch.Tensor]) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        B, S, P = batch["mzs"].shape
        mzs_flat = batch["mzs"].view(B * S, P)
        ints_flat = batch["ints"].view(B * S, P)
        nls_flat = batch["nls"].view(B * S, P)
        ce_flat = batch["ce"].view(B * S)
        add_flat = batch["adduct"].view(B * S)
        bpi_flat = batch["bpi"].view(B * S)

        h_s_flat, e_s_flat = self.spec_encoder(
            mzs_flat, ints_flat, nls_flat, ce_flat, add_flat, bpi_flat
        )
        h_s_all = h_s_flat.view(B, S, P, -1)
        e_s_all = e_s_flat.view(B, S, -1)
        return self.aggregator(h_s_all, e_s_all, batch["spec_mask"])

## Cell 6: Stage 1 Training — Contrastive Pre-Retrieval (Dual-Path InfoNCE Loss)

> **Review fix:** GLMR freezes a *pre-trained* ChemFormer. The template froze a randomly initialized encoder in both stages and never loaded weights, so Stage 1 aligned spectra to random molecule embeddings. The encoder is now frozen only when `RL77_MOL_ENCODER_CKPT` loads; otherwise both towers train jointly.

In [ ]:
# [Colab Cell 6] Stage 1: Dual-Path InfoNCE Contrastive Alignment with Intensity-Perturbed Negatives
def dual_path_infonce_loss(
    e_mol: torch.Tensor,
    e_spec: torch.Tensor,
    e_spec_perturbed: torch.Tensor,
    tau: float = CFG.TEMPERATURE,
) -> torch.Tensor:
    """
    GLMR Equations (3)-(5):
      - L_ms2mol: in-batch molecular negatives
      - L_mol2ms: positive spectrum vs. intensity-perturbed negative spectrum + in-batch spectra
      - L_pre = 0.5 * (L_ms2mol + L_mol2ms)
    """
    B = e_mol.size(0)
    labels = torch.arange(B, device=e_mol.device)

    # MS-to-Molecule alignment (Equation 4)
    logits_ms2mol = torch.matmul(e_spec, e_mol.T) / tau
    loss_ms2mol = F.cross_entropy(logits_ms2mol, labels)

    # Molecule-to-MS alignment with intensity-perturbed hard negatives (Equation 3)
    combined_spec = torch.cat([e_spec, e_spec_perturbed], dim=0)  # (2B, D)
    logits_mol2ms = torch.matmul(e_mol, combined_spec.T) / tau    # (B, 2B)
    loss_mol2ms = F.cross_entropy(logits_mol2ms, labels)

    return 0.5 * (loss_ms2mol + loss_mol2ms)


def load_pretrained_mol_encoder(model: "GLMRPipeline", ckpt_path: str) -> bool:
    """Loads a pre-trained ChemFormer encoder state_dict if one is provided. Returns True when loaded."""
    if not ckpt_path or not Path(ckpt_path).exists():
        ENV.log("No pre-trained ChemFormer checkpoint: the molecule encoder trains jointly in Stage 1.")
        return False
    state = torch.load(ckpt_path, map_location="cpu")
    model.mol_encoder.load_state_dict(state.get("state_dict", state))
    ENV.log(f"Loaded pre-trained ChemFormer encoder from {ckpt_path}; it stays frozen in Stage 1.")
    return True


def train_stage1_epoch(
    model: GLMRPipeline,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device = CFG.DEVICE,
    freeze_mol_encoder: bool = False,
    deadline: Optional["ENV.Deadline"] = None,
) -> float:
    """
    Updates SpectralEncoder + MultiSpectrumAggregator against the molecule encoder.

    GLMR freezes a *pre-trained* ChemFormer here. The original cell froze a randomly
    initialized encoder (no checkpoint was ever loaded), so spectra were aligned to random
    molecule embeddings. Freeze only when load_pretrained_mol_encoder() succeeded; otherwise
    both towers train jointly, CLIP-style.
    """
    model.train()
    if freeze_mol_encoder:
        model.mol_encoder.eval()
    for p in model.mol_encoder.parameters():
        p.requires_grad = not freeze_mol_encoder

    total_loss, n_batches = 0.0, 0
    for batch in loader:
        if deadline is not None and deadline.expired():
            break
        for k, v in batch.items():
            if isinstance(v, torch.Tensor):
                batch[k] = v.to(device)

        if freeze_mol_encoder:
            with torch.no_grad():
                _, e_m = model.mol_encoder(batch["mol_tokens_cls"])
        else:
            _, e_m = model.mol_encoder(batch["mol_tokens_cls"])

        # Positive multi-spectrum pass
        _, e_s, _ = model.encode_molecule_spectra(batch)

        # Construct intensity-perturbed spectral negative (GLMR Section 3.1)
        perturbed_batch = dict(batch)
        noise = torch.empty_like(batch["ints"]).uniform_(W("nb1.loss.perturb_low"), W("nb1.loss.perturb_high"))
        perturbed_ints = torch.clamp(batch["ints"] * noise, 0.0, 2.0)
        perturbed_ints[:, :, 0] = 2.0  # Keep precursor anchor intact
        perturbed_batch["ints"] = perturbed_ints
        _, e_s_neg, _ = model.encode_molecule_spectra(perturbed_batch)

        loss = dual_path_infonce_loss(e_m, e_s, e_s_neg, tau=CFG.TEMPERATURE)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / max(n_batches, 1)

## Cell 7: Stage 2 Training — Spec-Mol Cross-Fusion & Context-Aware Generation

> **Review fix:** When the 15 ppm window held fewer than K structures (most queries: ~275 library structures per Da), `retrieve_top_k_candidates` discarded the mass-matched candidates and returned a purely global top-K. In-window candidates are now kept first and global similarity only fills the remaining slots; ranking (Cell 8) also puts mass-consistent candidates ahead of the fill.

In [ ]:
# [Colab Cell 7] Stage 2: Context-Aware Generative Training (Cross-Fusion + Decoder)
@torch.no_grad()
def build_candidate_library_index(
    model: GLMRPipeline,
    candidate_smiles: List[str],
    tokenizer: SMILESTokenizer,
    batch_size: int = 256,
    device: torch.device = CFG.DEVICE,
) -> Tuple[List[str], np.ndarray, torch.Tensor]:
    """
    Encodes candidate database (train structures + PubChem/COCONUT) into normalized
    ChemFormer embeddings E^m and records exact neutral monoisotopic masses for ppm filtering.
    """
    from rdkit.Chem import Descriptors

    model.eval()
    valid_smiles, exact_masses, emb_batches = [], [], []

    for i in range(0, len(candidate_smiles), batch_size):
        chunk = candidate_smiles[i : i + batch_size]
        toks = []
        for smi in chunk:
            mol = Chem.MolFromSmiles(smi)
            if mol is None:
                continue
            valid_smiles.append(smi)
            exact_masses.append(Descriptors.ExactMolWt(mol))
            toks.append(tokenizer.encode(smi, add_cls=True))
        if not toks:
            continue
        tok_tensor = torch.stack(toks).to(device)
        _, e_m = model.mol_encoder(tok_tensor)
        emb_batches.append(e_m.cpu())

    library_embs = torch.cat(emb_batches, dim=0) if emb_batches else torch.empty(0, CFG.D_MODEL)
    return valid_smiles, np.asarray(exact_masses, dtype=np.float64), library_embs


def in_mass_window(masses: np.ndarray, neutral_mass: float, ppm_tol: float = CFG.MASS_TOLERANCE_PPM) -> np.ndarray:
    """Boolean mask of candidate exact masses within the ppm window (with the absolute floor)."""
    tol_da = max(neutral_mass * ppm_tol * 1e-6, CFG.MIN_TOLERANCE_DA)
    return np.abs(np.asarray(masses, dtype=np.float64) - neutral_mass) <= tol_da


@torch.no_grad()
def retrieve_top_k_candidates(
    e_s_batch: torch.Tensor,
    precursor_mzs: torch.Tensor,
    adducts: List[str],
    lib_masses: np.ndarray,
    lib_embs: torch.Tensor,
    top_k: int = CFG.TOP_K_CANDIDATES,
    ppm_tol: float = CFG.MASS_TOLERANCE_PPM,
    ionization_modes: Optional[List[str]] = None,
) -> Tuple[torch.Tensor, List[List[int]]]:
    """
    Pre-retrieves top-K candidate embeddings H^m_K per molecule using mass-constrained
    cosine similarity R(E^s, E^m_i). Falls back to global top-K if fewer than K mass hits exist.
    """
    device = e_s_batch.device
    lib_embs_dev = lib_embs.to(device)
    sim_matrix = torch.matmul(e_s_batch, lib_embs_dev.T)  # (B, N_lib)

    selected_embs = []
    selected_indices = []

    for b in range(e_s_batch.size(0)):
        prec_mz = float(precursor_mzs[b].item())
        mode = ionization_modes[b] if ionization_modes is not None else "positive"
        # Handles formate/acetate spelling, [M]+, and [2M+X] multimers (the old shift table did not).
        est_neutral_mass = neutral_mass_from_precursor(prec_mz, adducts[b], mode)

        mass_tol_da = max(est_neutral_mass * (ppm_tol * 1e-6), CFG.MIN_TOLERANCE_DA)
        mass_mask = np.abs(lib_masses - est_neutral_mass) <= mass_tol_da
        candidate_idx = np.where(mass_mask)[0]

        scores_b = sim_matrix[b]
        if len(candidate_idx) >= top_k:
            cand_t = torch.tensor(candidate_idx, dtype=torch.long, device=device)
            sub_scores = scores_b[cand_t]
            topk_local = torch.topk(sub_scores, k=top_k).indices
            top_idx = cand_t[topk_local]
        else:
            # Fewer than K structures in the mass window (the usual case at 15 ppm): keep all of them
            # first and fill the remaining slots from global similarity. The original code discarded the
            # mass-matched candidates here and returned a purely global top-K.
            k_total = min(top_k, len(lib_masses))
            cand_t = torch.tensor(candidate_idx, dtype=torch.long, device=device)
            in_window = cand_t[torch.argsort(scores_b[cand_t], descending=True)] if len(candidate_idx) else cand_t
            masked = scores_b.clone()
            masked[in_window] = float("-inf")
            fill = torch.topk(masked, k=k_total - len(in_window)).indices
            top_idx = torch.cat([in_window, fill])

        selected_indices.append(top_idx.cpu().tolist())
        selected_embs.append(lib_embs_dev[top_idx])

    return torch.stack(selected_embs, dim=0), selected_indices


def train_stage2_epoch(
    model: GLMRPipeline,
    loader: DataLoader,
    lib_masses: np.ndarray,
    lib_embs: torch.Tensor,
    optimizer: torch.optim.Optimizer,
    pad_id: int,
    device: torch.device = CFG.DEVICE,
    deadline: Optional["ENV.Deadline"] = None,
) -> float:
    """
    GLMR Section 3.2:
    Freezes both MolecularEncoder and SpectralEncoder; trains only SpecMolCrossFusion
    and ChemFormerMolecularDecoder via autoregressive cross-entropy loss L_gen (Eq. 8).
    """
    model.train()
    model.mol_encoder.eval()
    model.spec_encoder.eval()
    model.aggregator.eval()

    for module in (model.mol_encoder, model.spec_encoder, model.aggregator):
        for p in module.parameters():
            p.requires_grad = False

    total_loss, n_batches = 0.0, 0
    for batch in loader:
        if deadline is not None and deadline.expired():
            break
        for k, v in batch.items():
            if isinstance(v, torch.Tensor):
                batch[k] = v.to(device)

        with torch.no_grad():
            h_s_mol, e_s_mol, _ = model.encode_molecule_spectra(batch)
            h_m_k, _ = retrieve_top_k_candidates(
                e_s_mol,
                batch["precursor_mz"],
                batch["primary_adduct"],
                lib_masses,
                lib_embs,
                top_k=CFG.TOP_K_CANDIDATES,
                ionization_modes=batch["ionization_mode"],
            )

        # Fuse spectral sequence states with top-K retrieved molecular priors
        h_ca = model.cross_fusion(h_s_mol, h_m_k)

        # Autoregressive teacher forcing: predict y_{1..Q} from y_{0..Q-1}
        dec_tokens = batch["mol_tokens_dec"]
        inp_tokens = dec_tokens[:, :-1]
        tgt_tokens = dec_tokens[:, 1:]

        logits = model.mol_decoder(inp_tokens, h_ca)
        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            tgt_tokens.reshape(-1),
            ignore_index=pad_id,
        )

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(model.cross_fusion.parameters()) + list(model.mol_decoder.parameters()),
            1.0,
        )
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1

    return total_loss / max(n_batches, 1)

## Cell 8: Two-Stage GLMR Inference, Unimodal Re-Ranking & `inchikey14` Deduplication

> **Review fix:** The generated structure was always placed at rank 1 although the comment required a 15 ppm mass match; the mass check is implemented and the insertion rank comes from `W("nb1.rerank.generated_insert_rank")`. Ranking is factored into `assemble_ranked_candidates()` so validation can sweep `alpha` and the insertion rank on the same code path.

In [ ]:
# [Colab Cell 8] Stage 1 + Stage 2 Inference, Unimodal Re-Ranking, & Top-25 InChIKey14 Deduplication
def assemble_ranked_candidates(
    record: Dict,
    alpha_rerank: float,
    generated_insert_rank: int,
    top_n: int = CFG.TOP_N_SUBMISSION,
) -> List[str]:
    """
    Builds the deduplicated top-25 list for one molecule from the scores stored in `record`.
    Shared by inference and by the validation sweep that fits alpha / insert rank.
      - final = alpha * cos(E^m_+, E^m_i) + (1 - alpha) * cos(E^s, E^m_i) when the beam SMILES is valid
      - the generated structure enters at `generated_insert_rank` (1-based) only if its exact
        mass is within the ppm window of the precursor-derived neutral mass (0 disables it)
      - remaining slots are backfilled from global spectral similarity
    """
    stage1 = record["stage1_scores"]
    stage2 = record["stage2_scores"]
    final = stage1 if stage2 is None else alpha_rerank * stage2 + (1.0 - alpha_rerank) * stage1
    # Optional tiers first (notebook 4 formula priors: 0 = top formula, ...), then mass-consistent candidates,
    # then blended similarity (np.lexsort: last key is primary). Records without "priority" rank as before.
    in_window = np.asarray(record.get("in_window", np.zeros(len(final), dtype=bool)), dtype=bool)
    priority = np.asarray(record.get("priority", np.zeros(len(final), dtype=np.int64)))
    order = np.lexsort((-final, ~in_window, priority))

    seen_ik14, ranked = set(), []
    gen_smi, gen_ik14 = record["gen_smiles"], record["gen_ik14"]
    insert_gen = generated_insert_rank > 0 and record["gen_mass_ok"] and gen_smi is not None
    if insert_gen:
        seen_ik14.add(gen_ik14)  # reserve so a retrieved duplicate cannot take a second slot

    for i in order:
        canon_smi, ik14 = canonicalize_and_inchikey14(record["cand_smiles"][i])
        if canon_smi is not None and ik14 not in seen_ik14:
            seen_ik14.add(ik14)
            ranked.append(canon_smi)
        if len(ranked) >= top_n:
            break
    for smi in record["backfill_smiles"]:
        if len(ranked) >= top_n:
            break
        canon_smi, ik14 = canonicalize_and_inchikey14(smi)
        if canon_smi is not None and ik14 not in seen_ik14:
            seen_ik14.add(ik14)
            ranked.append(canon_smi)

    if insert_gen:
        pos = min(generated_insert_rank - 1, len(ranked))
        ranked.insert(pos, gen_smi)
    return ranked[:top_n]


@torch.no_grad()
def predict_and_rerank_test_molecules(
    model: GLMRPipeline,
    test_loader: DataLoader,
    tokenizer: SMILESTokenizer,
    lib_smiles: List[str],
    lib_masses: np.ndarray,
    lib_embs: torch.Tensor,
    alpha_rerank: float = W("nb1.rerank.alpha"),
    generated_insert_rank: int = W.int("nb1.rerank.generated_insert_rank"),
    device: torch.device = CFG.DEVICE,
    collect_records: Optional[List[Dict]] = None,
) -> pd.DataFrame:
    """
    Executes full GLMR two-stage inference per molecule_id:
      1. Multi-spectrum aggregation -> Stage 1 Pre-Retrieval of K=40 candidates.
      2. Spec-Mol Cross-Fusion + Beam Search (beam=5) -> generates refined SMILES y+.
      3. Encodes y+ via ChemFormerMolecularEncoder -> E^m_+ and re-ranks candidates via
         blended unimodal cosine similarity: alpha * cos(E^m_+, E^m_i) + (1 - alpha) * cos(E^s, E^m_i).
      4. Canonicalizes tautomers (RDKit 2026.03.3), deduplicates strictly by InChIKey14,
         and backfills to 25 unique structures per molecule_id.
    Per-molecule scores are appended to `collect_records` when given (validation sweeps).
    """
    from rdkit.Chem import Descriptors

    model.eval()
    lib_embs_dev = lib_embs.to(device)
    submission_rows = []

    for batch in test_loader:
        for k, v in batch.items():
            if isinstance(v, torch.Tensor):
                batch[k] = v.to(device)

        # Stage 1: Multi-spectrum pooling & top-K candidate pre-retrieval
        h_s_mol, e_s_mol, _ = model.encode_molecule_spectra(batch)
        h_m_k, topk_indices_batch = retrieve_top_k_candidates(
            e_s_mol,
            batch["precursor_mz"],
            batch["primary_adduct"],
            lib_masses,
            lib_embs,
            top_k=CFG.TOP_K_CANDIDATES,
            ionization_modes=batch["ionization_mode"],
        )

        # Stage 2: Cross-fusion & beam search generation of refined structures
        h_ca = model.cross_fusion(h_s_mol, h_m_k)
        gen_token_seqs = model.mol_decoder.beam_search_decode(
            h_ca, beam_size=CFG.BEAM_SIZE, max_len=W.int("nb1.decode.max_len")
        )

        for b, mol_id in enumerate(batch["molecule_id"]):
            cand_indices = topk_indices_batch[b]
            cand_embs = h_m_k[b]  # (K, D)
            stage1_scores = torch.matmul(cand_embs, e_s_mol[b]).cpu().numpy()  # (K,)

            gen_smiles_raw = tokenizer.decode(gen_token_seqs[b])
            canon_gen_smi, gen_ik14 = canonicalize_and_inchikey14(gen_smiles_raw)

            neutral = neutral_mass_from_precursor(
                float(batch["precursor_mz"][b]), batch["primary_adduct"][b], batch["ionization_mode"][b]
            )
            # If generated SMILES is chemically valid, compute unimodal similarity Eq. (9)
            stage2_scores, gen_mass_ok = None, False
            if canon_gen_smi is not None:
                gen_tok = tokenizer.encode(canon_gen_smi, add_cls=True).unsqueeze(0).to(device)
                _, e_m_plus = model.mol_encoder(gen_tok)  # (1, D)
                stage2_scores = torch.matmul(cand_embs, e_m_plus.squeeze(0)).cpu().numpy()  # (K,)
                gen_mass = Descriptors.ExactMolWt(Chem.MolFromSmiles(canon_gen_smi))
                gen_mass_ok = bool(in_mass_window(np.array([gen_mass]), neutral)[0])

            global_sims = torch.matmul(lib_embs_dev, e_s_mol[b])
            fallback_idx = torch.topk(global_sims, k=min(100, len(lib_smiles))).indices.cpu().tolist()

            record = {
                "molecule_id": str(mol_id),
                "cand_smiles": [lib_smiles[i] for i in cand_indices],
                "stage1_scores": stage1_scores,
                "stage2_scores": stage2_scores,
                "in_window": in_mass_window(lib_masses[cand_indices], neutral),
                "gen_smiles": canon_gen_smi,
                "gen_ik14": gen_ik14,
                "gen_mass_ok": bool(gen_mass_ok),
                "backfill_smiles": [lib_smiles[i] for i in fallback_idx],
            }
            if collect_records is not None:
                collect_records.append(record)

            unique_top25 = assemble_ranked_candidates(record, alpha_rerank, generated_insert_rank)
            submission_rows.append(
                {
                    "molecule_id": mol_id,
                    "smiles": ";".join(unique_top25[: CFG.TOP_N_SUBMISSION]),
                }
            )

    sub_df = pd.DataFrame(submission_rows, columns=["molecule_id", "smiles"])
    return sub_df


def validate_submission_file(sub_df: pd.DataFrame, expected_mol_ids: List[str]) -> None:
    """Automated submission gate checks required by CASMI 2026 rules (format of sample_submission.csv)."""
    assert list(sub_df.columns) == ["molecule_id", "smiles"], "Invalid column headers!"
    assert len(sub_df) == len(expected_mol_ids), "Row count mismatch with test molecule_ids!"
    assert sub_df["molecule_id"].is_unique, "Duplicate molecule_id rows!"
    assert set(sub_df["molecule_id"]) == set(expected_mol_ids), "Missing or unexpected molecule_id!"
    assert not sub_df["smiles"].isna().any(), "Null values found in smiles column!"
    for row in sub_df.itertuples():
        candidates = [c for c in str(row.smiles).split(";") if c]
        assert 1 <= len(candidates) <= 25, f"{row.molecule_id} has {len(candidates)} candidates!"
    print(f"Submission verified: {len(sub_df)} unique molecules, up to 25 deduplicated SMILES each.")

## Cell 9: End-to-End Orchestration on the Competition Data

The PDF's synthetic smoke test is replaced by the full run: stream `train.parquet` with a leakage-free timsTOF validation split, train Stage 1 and Stage 2 within their time budgets, evaluate MRR@25 on held-out molecules, cross-fit the re-ranking knobs, and infer `test.parquet`. Everything lands in `/kaggle/working`, which `tools/kaggle_pipeline.py` publishes as the `casmi26-rl77-nb1-glmr-outputs` dataset for notebooks 2–5.

| Output | Used by |
| --- | --- |
| `glmr_model.pt`, `checkpoints/glmr_stage1.pt` | notebooks 2, 3, 4, 5 |
| `library/library.parquet`, `library/lib_embs_fp16.npy` | notebooks 3, 5 |
| `split/val_inchikey14.txt`, `split/train_inchikey14.txt` | notebooks 2–5 (same validation molecules everywhere) |
| `metrics.json`, `fitted_weights.json`, `rerank_grid.json` | Evidently weights update |
| `submission.csv` | leaderboard submission |

In [ ]:
# [Colab Cell 9] End-to-end execution driver on the competition parquet files: train, validate, infer, save outputs
def save_glmr_checkpoint(model: GLMRPipeline, tokenizer: SMILESTokenizer, path: Path, extra: Optional[Dict] = None) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "state_dict": model.state_dict(),
            "vocab": tokenizer.vocab,
            "max_smiles_len": tokenizer.max_len,
            "adduct_vocab": ADDUCT_VOCAB,
            "weights_version": W.version,
            **(extra or {}),
        },
        path,
    )
    return path


def load_glmr_checkpoint(path: Path, device: torch.device = CFG.DEVICE) -> Tuple[GLMRPipeline, SMILESTokenizer]:
    """Rebuilds the GLMR pipeline saved by notebook 1 (used by notebooks 2-5)."""
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    tokenizer = SMILESTokenizer(max_len=ckpt.get("max_smiles_len", CFG.MAX_SMILES_LEN))
    if ckpt["vocab"] != tokenizer.vocab:
        raise ValueError("Tokenizer vocabulary differs from the checkpoint; rebuild notebook 1 outputs.")
    if ckpt.get("adduct_vocab", ADDUCT_VOCAB) != ADDUCT_VOCAB:
        raise ValueError("Adduct vocabulary differs from the checkpoint; rebuild notebook 1 outputs.")
    model = GLMRPipeline(tokenizer)
    model.load_state_dict(ckpt["state_dict"])
    return model.to(device).eval(), tokenizer


def save_library(out_dir: Path, lib_smiles: List[str], lib_ik14: List[str], lib_masses: np.ndarray, lib_embs: torch.Tensor) -> None:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({"smiles": lib_smiles, "inchikey14": lib_ik14, "exact_mass": lib_masses}).to_parquet(
        out_dir / "library.parquet", index=False
    )
    np.save(out_dir / "lib_embs_fp16.npy", lib_embs.numpy().astype(np.float16))


def load_library(lib_dir: Path) -> Tuple[List[str], List[str], np.ndarray, torch.Tensor]:
    lib = pd.read_parquet(Path(lib_dir) / "library.parquet")
    embs = torch.from_numpy(np.load(Path(lib_dir) / "lib_embs_fp16.npy").astype(np.float32))
    return lib["smiles"].tolist(), lib["inchikey14"].tolist(), lib["exact_mass"].to_numpy(np.float64), embs


def make_loader(df: pd.DataFrame, tokenizer: SMILESTokenizer, train: bool, batch_size: int = CFG.BATCH_SIZE) -> DataLoader:
    ds = MultiSpectrumMoleculeDataset(
        df, tokenizer,
        max_spectra_per_mol=CFG.MAX_SPECTRA_TRAIN if train else CFG.MAX_SPECTRA_INFER,
        is_train=train,
    )
    workers = CFG.NUM_WORKERS if train else 0
    return DataLoader(
        ds, batch_size=batch_size, shuffle=train, drop_last=train,
        num_workers=workers, persistent_workers=workers > 0,
    )


def predictions_from_submission(sub_df: pd.DataFrame) -> Dict[str, List[str]]:
    return {str(r.molecule_id): [s for s in str(r.smiles).split(";") if s] for r in sub_df.itertuples()}


@torch.no_grad()
def evaluate_stage1_retrieval(model, loader, lib_smiles, lib_ik14, lib_masses, lib_embs) -> Dict[str, float]:
    """Stage 1 only (no generation): MRR@25 and recall@K of the true inchikey14 in the K candidates."""
    model.eval()
    preds, gt, hits = {}, {}, []
    lib_embs_dev = lib_embs.to(CFG.DEVICE)
    for batch in loader:
        batch = {k: (v.to(CFG.DEVICE) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}
        _, e_s_mol, _ = model.encode_molecule_spectra(batch)
        _, idx_batch = retrieve_top_k_candidates(
            e_s_mol, batch["precursor_mz"], batch["primary_adduct"], lib_masses, lib_embs,
            top_k=CFG.TOP_K_CANDIDATES, ionization_modes=batch["ionization_mode"],
        )
        for b, mol_id in enumerate(batch["molecule_id"]):
            idx = idx_batch[b]
            scores = torch.matmul(lib_embs_dev[idx], e_s_mol[b]).cpu().numpy()
            fallback = torch.topk(torch.matmul(lib_embs_dev, e_s_mol[b]), k=min(100, len(lib_smiles))).indices.cpu().tolist()
            neutral = neutral_mass_from_precursor(
                float(batch["precursor_mz"][b]), batch["primary_adduct"][b], batch["ionization_mode"][b]
            )
            rec = {"cand_smiles": [lib_smiles[i] for i in idx], "stage1_scores": scores, "stage2_scores": None,
                   "in_window": in_mass_window(lib_masses[idx], neutral),
                   "gen_smiles": None, "gen_ik14": None, "gen_mass_ok": False,
                   "backfill_smiles": [lib_smiles[i] for i in fallback]}
            preds[mol_id] = assemble_ranked_candidates(rec, 0.0, 0)
            gt[mol_id] = batch["inchikey14"][b]
            hits.append(float(batch["inchikey14"][b] in {lib_ik14[i] for i in idx}))
    return {"mrr25": compute_casmi_mrr25(preds, gt), f"recall_at_{CFG.TOP_K_CANDIDATES}": float(np.mean(hits)), "n": len(gt)}


def sweep_rerank_knobs(
    records: List[Dict], gt: Dict[str, str], seed: int = CFG.SEED,
    prior_keys: Tuple[str, str] = ("nb1.rerank.alpha", "nb1.rerank.generated_insert_rank"),
) -> Dict:
    """Grid over alpha and generated-structure rank with 2-fold cross-fitting for an honest estimate.
    `prior_keys` name the Evidently weights the grid is compared against (notebook 5 passes its own)."""
    alphas = [round(a, 2) for a in np.linspace(0.0, 1.0, 11)]
    ranks = [0, 1, 3, 10, 25]
    grid = [(a, r) for a in alphas for r in ranks]

    def mrr(recs, a, r):
        preds = {rec["molecule_id"]: assemble_ranked_candidates(rec, a, r) for rec in recs}
        return compute_casmi_mrr25(preds, {rec["molecule_id"]: gt[rec["molecule_id"]] for rec in recs})

    rng = np.random.default_rng(seed)
    order = rng.permutation(len(records))
    folds = [[records[i] for i in order[0::2]], [records[i] for i in order[1::2]]]
    cv_scores = []
    for fit, held in ((folds[0], folds[1]), (folds[1], folds[0])):
        best = max(grid, key=lambda g: mrr(fit, *g))
        cv_scores.append(mrr(held, *best) * len(held))
    full_scores = {f"{a}|{r}": mrr(records, a, r) for a, r in grid}
    best_key = max(full_scores, key=full_scores.get)
    best_alpha, best_rank = best_key.split("|")
    prior = (W(prior_keys[0]), W.int(prior_keys[1]))
    return {
        "best_alpha": float(best_alpha),
        "best_generated_insert_rank": int(best_rank),
        "val_mrr25_prior": mrr(records, *prior),
        "val_mrr25_best_in_sample": full_scores[best_key],
        "val_mrr25_crossfit": float(sum(cv_scores) / len(records)),
        "grid": full_scores,
    }


def run_glmr_pipeline(out_dir: Optional[Path] = None) -> Dict:
    out_dir = Path(out_dir or CFG.OUTPUT_DIR)
    metrics: Dict = {"weights_version": W.version}
    tokenizer = SMILESTokenizer()

    # 1. Leakage-free real data (timsTOF validation blocks never seen in training)
    train_df, val_df, lib_df = load_competition_frames(CFG.DATA_DIR, CFG.N_TRAIN_MOLECULES, CFG.N_VAL_MOLECULES)
    train_df = train_df.dropna(subset=["normalized_smiles"]).reset_index(drop=True)
    (out_dir / "split").mkdir(parents=True, exist_ok=True)
    (out_dir / "split" / "val_inchikey14.txt").write_text("\n".join(sorted(val_df["inchikey14"].unique())) + "\n")
    (out_dir / "split" / "train_inchikey14.txt").write_text("\n".join(sorted(train_df["inchikey14"].unique())) + "\n")
    train_loader = make_loader(train_df, tokenizer, train=True)

    model = GLMRPipeline(tokenizer).to(CFG.DEVICE)
    pretrained = load_pretrained_mol_encoder(model, CFG.MOL_ENCODER_CKPT)

    # 2. Stage 1: dual-path InfoNCE (spectrum <-> molecule)
    s1_params = list(model.spec_encoder.parameters()) + list(model.aggregator.parameters())
    if not pretrained:
        s1_params += list(model.mol_encoder.parameters())
    opt_s1 = torch.optim.AdamW(s1_params, lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    deadline = ENV.Deadline(CFG.STAGE1_MINUTES)
    history = []
    for epoch in range(CFG.EPOCHS_STAGE1):
        loss = train_stage1_epoch(model, train_loader, opt_s1, freeze_mol_encoder=pretrained, deadline=deadline)
        history.append({"stage": 1, "epoch": epoch, "loss": loss})
        ENV.log(f"Stage 1 epoch {epoch}: InfoNCE {loss:.4f} ({deadline.remaining_min():.1f} min left)")
        if deadline.expired():
            break
    save_glmr_checkpoint(model, tokenizer, CFG.CHECKPOINT_DIR / "glmr_stage1.pt")

    # 3. Retrieval library: every train.parquet structure, embedded by the Stage 1 molecule encoder
    lib_smiles, lib_masses, lib_embs = build_candidate_library_index(model, lib_df["smiles"].tolist(), tokenizer)
    smi_to_ik14 = dict(zip(lib_df["smiles"], lib_df["inchikey14"]))
    lib_ik14 = [smi_to_ik14[s] for s in lib_smiles]
    save_library(out_dir / "library", lib_smiles, lib_ik14, lib_masses, lib_embs)
    ENV.log(f"Library: {len(lib_smiles):,} structures")

    rng = np.random.default_rng(CFG.SEED)
    val_ids = sorted(val_df["inchikey14"].unique())
    eval_ids = set(rng.choice(val_ids, size=min(CFG.N_VAL_EVAL, len(val_ids)), replace=False).tolist())
    eval_df = val_df[val_df["inchikey14"].isin(eval_ids)].reset_index(drop=True)
    # Same molecules as the two-stage evaluation below, so the Stage 1 vs Stage 2 numbers are comparable.
    metrics["val_stage1"] = evaluate_stage1_retrieval(
        model, make_loader(eval_df, tokenizer, train=False, batch_size=32), lib_smiles, lib_ik14, lib_masses, lib_embs
    )
    ENV.log(f"Validation Stage 1 only: {metrics['val_stage1']}")

    # 4. Stage 2: cross-fusion + decoder with frozen encoders
    opt_s2 = torch.optim.AdamW(
        list(model.cross_fusion.parameters()) + list(model.mol_decoder.parameters()),
        lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY,
    )
    deadline = ENV.Deadline(CFG.STAGE2_MINUTES)
    for epoch in range(CFG.EPOCHS_STAGE2):
        loss = train_stage2_epoch(model, train_loader, lib_masses, lib_embs, opt_s2, tokenizer.pad_id, deadline=deadline)
        history.append({"stage": 2, "epoch": epoch, "loss": loss})
        ENV.log(f"Stage 2 epoch {epoch}: token CE {loss:.4f} ({deadline.remaining_min():.1f} min left)")
        if deadline.expired():
            break
    save_glmr_checkpoint(model, tokenizer, out_dir / "glmr_model.pt")
    pd.DataFrame(history).to_csv(out_dir / "train_log.csv", index=False)

    # 5. Full two-stage validation on held-out timsTOF molecules + re-ranking knob sweep
    records: List[Dict] = []
    val_sub = predict_and_rerank_test_molecules(
        model, make_loader(eval_df, tokenizer, train=False, batch_size=16), tokenizer,
        lib_smiles, lib_masses, lib_embs, collect_records=records,
    )
    gt = ground_truth_ik14(eval_df)
    metrics["val_two_stage_mrr25"] = compute_casmi_mrr25(predictions_from_submission(val_sub), gt)
    metrics["val_generated_valid_rate"] = float(np.mean([r["gen_smiles"] is not None for r in records]))
    metrics["val_generated_mass_ok_rate"] = float(np.mean([r["gen_mass_ok"] for r in records]))
    sweep = sweep_rerank_knobs(records, gt)
    metrics["rerank_sweep"] = {k: v for k, v in sweep.items() if k != "grid"}
    val_sub.to_csv(out_dir / "val_predictions.csv", index=False)
    ENV.write_json(out_dir / "rerank_grid.json", sweep["grid"])
    ENV.write_json(out_dir / "fitted_weights.json", {
        "notebook": "nb1",
        "weights": {
            "nb1.rerank.alpha": sweep["best_alpha"],
            "nb1.rerank.generated_insert_rank": sweep["best_generated_insert_rank"],
        },
        "evidence": {
            "metric": "MRR@25",
            "split": f"{len(records)} held-out timsTOF molecules, 2-fold cross-fit",
            "prior": sweep["val_mrr25_prior"],
            "fitted_crossfit": sweep["val_mrr25_crossfit"],
        },
    })
    ENV.log(f"Validation two-stage MRR@25 {metrics['val_two_stage_mrr25']:.4f}; sweep {metrics['rerank_sweep']}")

    # 6. Test inference with the active (Evidently) weights
    test_df = pd.read_parquet(CFG.DATA_DIR / "test.parquet")
    sub_df = predict_and_rerank_test_molecules(
        model, make_loader(test_df, tokenizer, train=False, batch_size=16), tokenizer,
        lib_smiles, lib_masses, lib_embs,
    )
    validate_submission_file(sub_df, test_df["molecule_id"].unique().tolist())
    sub_df.to_csv(CFG.SUBMISSION_PATH, index=False)

    ENV.write_json(out_dir / "metrics.json", metrics)
    ENV.write_run_manifest("nb1", out_dir, W.version, {"metrics": metrics})
    ENV.log(f"Notebook 1 outputs written to {out_dir}")
    return metrics


if __name__ == "__main__":
    NB1_METRICS = run_glmr_pipeline()

## Key Architectural Highlights Aligned with the Rubric

- **Positive-Mode Electron-Mass Alignment (Cell 3):** Subtracts `0.000548` Da from `enveda-180` positive-mode training spectra so fragment and precursor *m/z* values match the hidden test set calibration.
- **Multi-Spectrum Attention Pooling (Cell 5):** Rather than predicting per spectrum and heuristically merging ranks, `MultiSpectrumMoleculeAggregator` applies cross-spectrum self-attention across all collision energies (`collision_energy_ev`) and `base_peak_intensity` levels for a given `molecule_id` to form both a global vector $\mathbf{E}^s_{\text{mol}}$ and sequence query matrix $\mathbf{H}^s_{\text{mol}}$.
- **Strict `inchikey14` Deduplication (Cell 8):** Every generated and retrieved SMILES candidate is passed through `rdMolStandardize.TautomerEnumerator().Canonicalize` and deduplicated by its 14-character connectivity block (`inchikey14`), ensuring zero wasted rank slots among the 25 predictions per molecule.